# EuroCup API — Explorare date U-BT

Notebook de explorare înainte de a scrie fetcherii definitivi.  
Rulează celulele în ordine — fiecare secțiune testează un endpoint.

**Competiție:** `'U'` = EuroCup &nbsp;|&nbsp; `'E'` = Euroleague  
**Season:** numărul anului de start (ex: `2025` = sezonul 2025-26)

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.float_format', '{:.2f}'.format)

# Add project root to path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from euroleague_api.game_metadata import GameMetadata
from euroleague_api.player_stats  import PlayerStats
from euroleague_api.team_stats    import TeamStats
from euroleague_api.standings     import Standings
from euroleague_api.game_stats    import GameStats

COMPETITION = 'U'     # EuroCup
SEASON      = 2025    # 2025-26
TEAM_CODE   = 'UBT'   # ajustează dacă e diferit după prima rulare

print('Setup OK')

## 1. Game Codes — ce meciuri există în sezon

In [ ]:
gm = GameMetadata(COMPETITION)

# Toate game code-urile din sezon
df_codes = gm.get_gamecodes_season(season=SEASON)
print(f'Total meciuri în sezon: {len(df_codes)}')
print(f'Coloane: {list(df_codes.columns)}')
df_codes.head(10)

## 2. Game Metadata — detalii toate meciurile din sezon

In [ ]:
df_games_all = gm.get_game_metadata_single_season(season=SEASON)
print(f'Total rânduri: {len(df_games_all)}')
print(f'Coloane ({len(df_games_all.columns)}): {list(df_games_all.columns)}')
df_games_all.head(3)

In [ ]:
# Toate codurile de echipă — ca să verificăm cum apare U-BT
home_codes = df_games_all['homecode'].unique() if 'homecode' in df_games_all.columns else []
away_codes = df_games_all['awaycode'].unique() if 'awaycode' in df_games_all.columns else []
all_codes  = sorted(set(list(home_codes) + list(away_codes)))
print('Toate codurile echipelor:')
print(all_codes)

In [ ]:
# !! Actualizează TEAM_CODE de mai sus dacă e diferit de 'UBT'

# Filtrare meciuri U-BT
mask_home = df_games_all.get('homecode', pd.Series()).str.upper() == TEAM_CODE
mask_away = df_games_all.get('awaycode', pd.Series()).str.upper() == TEAM_CODE
df_ubt = df_games_all[mask_home | mask_away].copy()

print(f'Meciuri U-BT găsite: {len(df_ubt)}')
df_ubt[['gamenumber','round','date','hometeam','awayteam','homescore','awayscore','arena']].head(20)

## 3. Game Metadata — un singur meci (detaliu)

In [ ]:
# Ia primul game_code al U-BT
if len(df_ubt) > 0:
    first_code = int(df_ubt.iloc[0]['gamenumber'])
    print(f'Primul meci U-BT: game_code={first_code}')
    
    df_one = gm.get_game_metadata(season=SEASON, gamecode=first_code)
    print(f'Coloane: {list(df_one.columns)}')
    df_one.T  # transpus pentru lizibilitate
else:
    print('Nu s-au găsit meciuri U-BT. Verifică TEAM_CODE.')

## 4. Player Stats — statistici jucători sezon

In [ ]:
ps = PlayerStats(COMPETITION)

# Per game averages — toți jucătorii din competiție
df_ps_all = ps.get_player_stats_single_season(
    endpoint='players',
    season=SEASON,
    statistic_mode='PerGame'
)
print(f'Total jucători: {len(df_ps_all)}')
print(f'Coloane ({len(df_ps_all.columns)}): {list(df_ps_all.columns)}')
df_ps_all.head(3)

In [ ]:
# Coloane disponibile cu valori non-null
print('Coloane cu date (non-null > 50%):')
threshold = len(df_ps_all) * 0.5
good_cols = [c for c in df_ps_all.columns if df_ps_all[c].notna().sum() > threshold]
print(good_cols)

In [ ]:
# Filtrare jucători U-BT
# Încearcă diferite variante de coloană team
team_col = None
for c in ['teamcode', 'team_code', 'clubcode', 'club_code', 'TeamCode']:
    if c in df_ps_all.columns:
        team_col = c
        break

if team_col:
    print(f'Coloana echipă: {team_col}')
    print('Valori unice:', df_ps_all[team_col].unique()[:10])
    df_ubt_players = df_ps_all[df_ps_all[team_col].str.upper() == TEAM_CODE]
    print(f'\nJucători U-BT: {len(df_ubt_players)}')
    df_ubt_players
else:
    print('Nu s-a găsit coloana echipă. Coloane disponibile:')
    print(list(df_ps_all.columns))

In [ ]:
# Totals în loc de PerGame
df_ps_totals = ps.get_player_stats_single_season(
    endpoint='players',
    season=SEASON,
    statistic_mode='Accumulated'
)
print(f'Totals — rânduri: {len(df_ps_totals)}')

if team_col and team_col in df_ps_totals.columns:
    df_ubt_totals = df_ps_totals[df_ps_totals[team_col].str.upper() == TEAM_CODE]
    print(f'Jucători U-BT (totals): {len(df_ubt_totals)}')
    df_ubt_totals

## 5. Player Stats Leaders — top marcatori etc.

In [ ]:
# Top marcatori U-BT în competiție
df_leaders = ps.get_player_stats_leaders_single_season(
    season=SEASON,
    club_code=TEAM_CODE,
    stat_category='Points',
    top_n=20,
    statistic_mode='PerGame'
)
print(f'Leaders returned: {len(df_leaders)}')
df_leaders.head(15)

## 6. Team Stats — statistici echipă

In [ ]:
ts = TeamStats(COMPETITION)

df_team = ts.get_team_stats_single_season(
    endpoint='teams',
    season=SEASON,
    statistic_mode='PerGame'
)
print(f'Total echipe: {len(df_team)}')
print(f'Coloane: {list(df_team.columns)}')

# Găsim U-BT
team_col_t = None
for c in ['teamcode', 'team_code', 'clubcode', 'TeamCode']:
    if c in df_team.columns:
        team_col_t = c
        break

if team_col_t:
    df_ubt_team = df_team[df_team[team_col_t].str.upper() == TEAM_CODE]
    print(f'\nU-BT team stats:')
    df_ubt_team.T
else:
    df_team

## 7. Box Score — statistici meci individual

In [ ]:
gs = GameStats(COMPETITION)

if len(df_ubt) > 0:
    game_code = int(df_ubt.iloc[0]['gamenumber'])
    print(f'Box score pentru game_code={game_code}...')
    
    # Player box score
    df_box = gs.get_game_stats(season=SEASON, gamecode=game_code, endpoint='PlayerStats')
    print(f'Coloane: {list(df_box.columns)}')
    df_box
else:
    print('Nu există meciuri U-BT pentru a face box score.')

In [ ]:
# Toate endpoint-urile disponibile pentru game stats
import inspect
print(inspect.signature(gs.get_game_stats))

# Încearcă și team stats per meci
if len(df_ubt) > 0:
    df_box_team = gs.get_game_stats(season=SEASON, gamecode=game_code, endpoint='TeamStats')
    print('\nTeam box score:')
    df_box_team

## 8. Standings

In [ ]:
s = Standings(COMPETITION)

# Încearcă round 1, 5, 10 — unele pot returna 403 dacă runda nu e finalizată
for rnd in [1, 5, 10]:
    try:
        df_st = s.get_standings(season=SEASON, round_number=rnd)
        print(f'Round {rnd}: {len(df_st)} echipe')
        print(f'Coloane: {list(df_st.columns)}')
        display(df_st[['cname','played','wins','losses']].head(5))
        break
    except Exception as e:
        print(f'Round {rnd}: {e}')

## 9. Rezumat coloane disponibile

Celulă de sinteză — rulează după ce ai explorat datele.

In [ ]:
# Mapare coloane raw → schema finală games.csv
print('=== GAMES — mapare coloane ===')
games_map = {
    'competition':  'EuroCup (hardcoded)',
    'season':       'SEASON (hardcoded)',
    'game_code':    'gamenumber',
    'round':        'round',
    'date':         'date',
    'home_team':    'hometeam',
    'home_code':    'homecode',
    'away_team':    'awayteam',
    'away_code':    'awaycode',
    'score_home':   'homescore',
    'score_away':   'awayscore',
    'venue':        'arena',
    'city':         'city',
    'attendance':   'capacity (sau alt câmp)',
    'phase':        'phaseseason',
}

if 'df_games_all' in dir():
    available = set(df_games_all.columns.str.lower())
    for target, source in games_map.items():
        src_low = source.lower().split(' ')[0]
        found = '✓' if src_low in available else '✗ LIPSĂ'
        print(f'  {found}  {target:15} ← {source}')
else:
    print('Rulează mai întâi celulele de mai sus.')

In [ ]:
print('=== PLAYER STATS — mapare coloane ===')
players_map = {
    'player_id':    'playerid',
    'player_name':  'player',
    'games_played': 'gamesplayed',
    'minutes':      'timeplayed',
    'points':       'points',
    'rebounds':     'totalrebounds',
    'assists':      'assists',
    'steals':       'steals',
    'blocks':       'blocksfavour',
    'turnovers':    'turnovers',
    'fg2_made':     'fieldgoals2made',
    'fg2_att':      'fieldgoals2att',
    'fg3_made':     'fieldgoals3made',
    'fg3_att':      'fieldgoals3att',
    'ft_made':      'freethrowsmade',
    'ft_att':       'freethrowsatt',
    'efficiency':   'valuation',
}

if 'df_ps_all' in dir():
    available = set(df_ps_all.columns.str.lower())
    for target, source in players_map.items():
        found = '✓' if source.lower() in available else '✗ LIPSĂ'
        print(f'  {found}  {target:15} ← {source}')
else:
    print('Rulează celulele de player stats mai întâi.')

## 10. Export rapid CSV (test)

Salvează datele brute în `data/raw/eurocup/` pentru inspecție.

In [ ]:
import json
from datetime import datetime, timezone

raw_dir = ROOT / 'data' / 'raw' / 'eurocup'
raw_dir.mkdir(parents=True, exist_ok=True)

saved = []

if 'df_games_all' in dir() and len(df_games_all) > 0:
    p = raw_dir / f'games_raw_{SEASON}.csv'
    df_games_all.to_csv(p, index=False)
    saved.append(str(p))

if 'df_ps_all' in dir() and len(df_ps_all) > 0:
    p = raw_dir / f'player_stats_raw_{SEASON}.csv'
    df_ps_all.to_csv(p, index=False)
    saved.append(str(p))

if 'df_ubt' in dir() and len(df_ubt) > 0:
    p = raw_dir / f'ubt_games_{SEASON}.csv'
    df_ubt.to_csv(p, index=False)
    saved.append(str(p))

print('Fișiere salvate:')
for f in saved:
    print(f'  {f}')